# Multimodal Agents

**Level:** Advanced · **Time:** 60 min

Agents that can see, hear, and click are incredibly powerful, but introduce entirely new classes of bugs and security vulnerabilities.

In this notebook, we will simulate two critical multimodal patterns:
1. **Structured Vision Extraction:** Forcing an agent to extract strict JSON from an image and rejecting hallucinated math.
2. **Computer Use (Stale State Prevention):** An agent predicts X/Y coordinates for a mouse click, but the Tool Gateway blocks it because the screen state changed.

---
## Pattern 1: Structured Vision Extraction

We pass a blurry receipt to a vision model. It extracts the items, but hallucinates the total. Our deterministic validation layer catches the math error before it can be saved to the database.

In [ ]:
import json

# Simulated Vision Model Output (with a hallucinated total)
def mock_vision_model(image_path: str):
    print(f"[Vision LLM] Analyzing {image_path}...")
    # The receipt says $10 + $5, but the blurry total looks like $100
    return {
        "merchant": "Acme Corp",
        "items": [
            {"name": "Widget", "price": 10.00},
            {"name": "Gadget", "price": 5.00}
        ],
        "total": 100.00  # Hallucination! Should be 15.00
    }

def process_receipt(image_path: str):
    print("\n--- Processing Receipt ---")
    extracted_json = mock_vision_model(image_path)
    
    # Deterministic Validation Layer
    calculated_total = sum(item["price"] for item in extracted_json["items"])
    
    if calculated_total != extracted_json["total"]:
        print(f"[Validation Failed] Math mismatch! LLM claims total is ${extracted_json['total']}, but items sum to ${calculated_total}.")
        print("[System] Rejecting payload. Requesting human review.")
        return False
        
    print("[Validation Passed] Saving to database.")
    return True

process_receipt("blurry_receipt.jpg")



--- Processing Receipt ---
[Vision LLM] Analyzing blurry_receipt.jpg...
[Validation Failed] Math mismatch! LLM claims total is $100.0, but items sum to $15.0.
[System] Rejecting payload. Requesting human review.


---
## Pattern 2: Computer Use (Stale State Prevention)

An agent looks at a screenshot and decides to click the "Submit" button at `X: 500, Y: 800`. However, in the 2 seconds it took the LLM to process, an ad popped up. The Tool Gateway must prevent the click.

In [ ]:
import time

# Simulated UI State
current_screen_hash = "hash_v1_clean_screen"

def mock_computer_use_agent(screenshot_hash: str):
    print(f"[Agent] Looking at screen ({screenshot_hash})...")
    print("[Agent] I found the 'Submit' button.")
    # The agent decides to click X:500, Y:800 based on the screenshot it saw.
    # It must pass the hash of the screenshot it looked at back to the tool.
    return {"action": "click", "x": 500, "y": 800, "reference_hash": screenshot_hash}

def execute_mouse_click(x: int, y: int, reference_hash: str):
    global current_screen_hash
    
    print(f"\n[Tool Gateway] Agent requested to click ({x}, {y}) based on {reference_hash}.")
    
    # Critical Security Check: Stale State Prevention
    if reference_hash != current_screen_hash:
        print(f"[Tool Gateway] ❌ BLOCKED! The screen state has changed (Current: {current_screen_hash}).")
        print("[Tool Gateway] The agent might be clicking a pop-up ad or a moved button. Forcing agent to look again.")
        return False
        
    print(f"[Tool Gateway] ✅ SUCCESS. Clicking ({x}, {y}).")
    return True

# 1. Take screenshot
screenshot = current_screen_hash

# 2. A pop-up ad appears while the agent is thinking!
time.sleep(1)
print("\n[System Event] Pop-up ad appeared!")
current_screen_hash = "hash_v2_popup_ad_active"

# 3. The agent outputs its decision based on the OLD screenshot
decision = mock_computer_use_agent(screenshot)

# 4. The Tool Gateway intercepts the click
execute_mouse_click(decision["x"], decision["y"], decision["reference_hash"])



[System Event] Pop-up ad appeared!
[Agent] Looking at screen (hash_v1_clean_screen)...
[Agent] I found the 'Submit' button.

[Tool Gateway] Agent requested to click (500, 800) based on hash_v1_clean_screen.
[Tool Gateway] ❌ BLOCKED! The screen state has changed (Current: hash_v2_popup_ad_active).
[Tool Gateway] The agent might be clicking a pop-up ad or a moved button. Forcing agent to look again.
